# Lab Kit L1 · Your robot lab in a browser tab

**Video:** Lab Kit L1 on [Foundation Models for Robotics](https://www.youtube.com/@FoundationModelsForRobotics)
**You need:** a Google account for Colab, or Python 3.10+ on your machine. No GPU. About 15 minutes.

Every lesson in the series comes with a lab like this one. This first one is a tour: load a robot, look inside it, move it, take pictures of it. Run the cells top to bottom with Shift+Enter.

In [ ]:
# Setup: works in Google Colab and locally (after `pip install -e .` in the repo root)
import os, sys
if "google.colab" in sys.modules:
    os.environ["MUJOCO_GL"] = "egl"
    if not os.path.exists("fmr-robot-learning-labs"):
        !git clone -q https://github.com/kandapagari/fmr-robot-learning-labs.git
    %cd fmr-robot-learning-labs
    !pip install -q -e .
import numpy as np, matplotlib.pyplot as plt, mediapy as media, mujoco
import fmr6


def show_video(frames, fps=30):
    """Play frames inline. Without ffmpeg (some local installs) show a strip of stills instead."""
    try:
        media.show_video(frames, fps=fps)
    except RuntimeError:
        media.show_images(frames[:: max(1, len(frames) // 5)], columns=5)

## 1 · Load a robot

`fmr6.Robot(scene, arm=...)` loads a MuJoCo model. The scene is `"arm"` (robot alone) or `"table"` (robot, table, cup and sponge). The arm is `"fr3"`, a Franka FR3 with the Franka Hand, or `"fmr6"`, the six-axis arm drawn in the videos.

In [ ]:
robot = fmr6.Robot("table", arm="fr3")
media.show_image(robot.render(640, 400, camera={"lookat": [0.45, 0.0, 0.45], "distance": 2.0, "azimuth": 140, "elevation": -18}))

## 2 · Look inside

A robot model is a list of joints, each with limits, and a list of actuators (the motors that drive them). The FR3 has seven joints and one more actuator for the gripper.

In [ ]:
for name, lo, hi in zip(robot.joints, np.degrees(robot.lo), np.degrees(robot.hi)):
    print(f"{name:12s} {lo:7.1f}° … {hi:6.1f}°")
print("actuators:", [robot.model.actuator(i).name for i in range(robot.model.nu)])
print("pose now (degrees):", np.degrees(robot.q()).round(1))

## 3 · Move one joint

`robot.set_q(q)` places the arm at joint angles `q` (radians) and updates where every link is. Nothing is simulated yet: this is pure geometry, the same thing you'd do with a protractor on a drawing. Here we turn the base joint and then the elbow (joint 4).

In [ ]:
q = robot.ready.copy()
views = []
for j, deg in [(None, 0), (0, 40), (3, 35)]:
    if j is not None:
        q[j] += np.radians(deg)
    robot.set_q(q)
    views.append(robot.render(320, 240, camera={"lookat": [0.45, 0.0, 0.45], "distance": 2.0, "azimuth": 140, "elevation": -18}))
media.show_images(views, titles=["ready", "joint 1 +40°", "then joint 4 +35°"])

## 4 · Where is the hand?

The point that matters most is the **tool centre point** (TCP): the spot between the fingertips. `robot.tcp()` returns its position in metres and its orientation as a 3×3 rotation matrix. On the FR3 the hand points along the matrix's third column.

In [ ]:
robot.set_q(robot.ready)
pos, R = robot.tcp()
print("TCP position (m):", pos.round(3))
print("hand points along:", R[:, robot.tool_axis].round(2), " (straight down is [0, 0, -1])")

## 5 · Two arms, one table

FMR-6 has six joints and a simpler shape, which makes the maths in the videos easy to draw. The FR3 has seven, which gives it one spare degree of freedom: it can move its elbow while the hand stays still. Same table, same objects, same `Robot` methods.

In [ ]:
pics = []
for arm in ("fr3", "fmr6"):
    r = fmr6.Robot("table", arm=arm)
    pics.append(r.render(400, 300, camera={"lookat": [0.45, 0.0, 0.45], "distance": 2.0, "azimuth": 140, "elevation": -18}))
media.show_images(pics, titles=["Franka FR3 (7 joints)", "FMR-6 (6 joints)"])

## 6 · A short animation

A video is a list of images. Sweep joint 1 back and forth and render a frame each time.

In [ ]:
frames = []
for t in np.linspace(0, 2 * np.pi, 60):
    q = robot.ready.copy()
    q[0] = 0.6 * np.sin(t)
    robot.set_q(q)
    frames.append(robot.render(480, 300, camera={"lookat": [0.45, 0.0, 0.45], "distance": 2.0, "azimuth": 140, "elevation": -18}))
show_video(frames, fps=30)

## 7 · Try it

1. Find joint angles that put the fingertips just above the cup (hint: start from `robot.ready`, change one joint at a time, print `robot.tcp()[0]`). Then compare with the cup's position: `robot.data.body("cup").xpos`.
2. Set joint 4 to a value outside its limits and call `set_q`. What happens? (Nothing stops you: `set_q` is geometry only. Limits matter once we simulate physics in L3.)
3. Change `camera=` to look at the robot from above, then from the side.
4. Open the first real lab: [S0E1 · Small errors don't stay small](https://colab.research.google.com/github/kandapagari/fmr-robot-learning-labs/blob/main/labs/s0e1-drift/s0e1_drift.ipynb).

**Next:** L2 drives the arm to a point you choose instead of to joint angles.

In [ ]:
# your experiments here